## 假设验证实验设计

---

### 假设总览

| 假设 | 核心问题 | 验证方法 |
|-----|---------|---------|
| **H1** | 目标冲突 | 分析梯度方向一致性 |
| **H2** | 权重过大 | 测试更小的权重 |
| **H3** | 采样偏差 | 测试不同采样率 |
| **H4** | 时机不当 | 延迟启用同步损失 |
| **H5** | 距离度量问题 | 尝试其他度量 |



---

## H2 实验整理

### 1. H2 假设定义

| 假设 | 核心问题 | 验证方法 |
|:---:|:---|:---|
| **H2** | **权重过大**：`sync_weight=0.1` 太大，导致同步损失压过了交叉熵损失 | 测试更小的权重 |

### 2. 实验设计

| 实验 | sync_weight | desync_weight | 目的 |
|:---:|:---:|:---:|:---|
| A (原始) | 0.1 | 0.1 | 基准 |
| **H2a** | 0.01 | 0.01 | 降低10倍 |
| **H2b** | 0.001 | 0.001 | 降低100倍 |
| H2c | 0.05 | 0.0 | 仅sync |
| H2d | 0.0 | 0.05 | 仅desync |

### 3. 代码库现状

在项目文件中**没有**独立的 H2 配置文件（如 `experiment_H2a_small_weight.yaml`），你需要：

- **手动修改** `chaotic_config.yaml` 中的 `synchronization` 配置
- 或**创建**专门的实验配置文件

H2 实验是**纯配置修改**，不需要改代码（P0优先级）。

### 4. 已完成的实验结果

根据之前对话，H2 实验**已经完成**：

| 实验 | sync_weight | Accuracy | Loss | Confidence | Diversity |
|:---:|:---:|:---:|:---:|:---:|:---:|
| A (原始) | 0.1 | **94.07%** | 0.1514 | 0.6790 | 0.2561 |
| **H2a** | 0.01 | **96.61%** | 0.1306 | 0.5680 | 0.1317 |
| **H2b** | 0.001 | **96.61%** | **0.1078** | **0.9994** | 0.2486 |
| B (无同步) | 0 | 96.61% | 0.1552 | 0.9692 | 0.2439 |

---

## 5. 理想实验结果 vs 实际结论

### ✅ 假设 H2 被确认

| 期望结果 | 实际观察 | 结论 |
|:---|:---|:---|
| H2a/H2b 准确率提升到 ~96% | ✅ **确认**：从 94.07% → 96.61% | 权重 0.1 确实太大 |

### ⚠️ 但同步损失**没有带来增益**

```
H2a = H2b = B = 96.61%
```

- 降低权重后，性能**恢复到基线**，但**没有超越无同步损失的配置**
- 置信度分析显示 H2b（0.001）的置信度最高（0.9994），可能有轻微正则化效果

---

## 6. 最终结论

| 问题 | 答案 |
|:---|:---|
| 权重 0.1 太大吗？ | ✅ **是的，确认** |
| 降低权重能改善吗？ | ✅ **是的，恢复到基线水平** |
| 同步损失有帮助吗？ | ❌ **没有明显帮助**（等于基线） |
| 最佳同步权重是？ | **0.001 或直接禁用** |

### 推荐配置

**选项1（最简单）**：直接禁用同步损失
```yaml
synchronization:
  enabled: false
```

**选项2**：保留极小权重（如果想保留理论框架）
```yaml
synchronization:
  enabled: true
  sync_weight: 0.001
  desync_weight: 0.001
```

---

## Run H2: Change sync_weight

| 实验 | sync_weight | desync_weight |
| --- | --- | --- |
| H2-a | 0.01 | 0.01 |
| H2-b | 0.001 | 0.001 |
| H2-c | 0.05 | 0.0 |
| H2-d | 0.0 | 0.05 |

In [1]:
# H2a
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --epochs 100 \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2a \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2a/experiment_H2a_small_weight.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

In [2]:
# H2b
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --epochs 100 \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2b \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2b/experiment_H2b_tiny_weight.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

In [3]:
# H2c
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --epochs 100 \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

In [4]:
# H2d
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --epochs 100 \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2d \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2d/experiment_H2d_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

In [5]:
# H2e
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --epochs 100 \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2e \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2e/experiment_H2e.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

## H2 实验完整分析（全部数据）

### 全部结果汇总

| 实验 | sync_weight | desync_weight | Acc | Loss |
|:---:|:---:|:---:|:---:|:---:|
| H2-f | 0.03 | 0.0 | 94.07% | 0.2948 |
| **H2-c** | **0.05** | **0.0** | **96.61%** ⭐ | **0.1478** |
| H2-g | 0.07 | 0.0 | 94.92% | 0.2459 |
| H2-e | 0.1 | 0.0 | 93.22% | 0.2253 |
| H2-a | 0.01 | 0.01 | 94.07% | 0.2734 |
| H2-b | 0.001 | 0.001 | 94.92% | 0.1847 |
| H2-d | 0.0 | 0.05 | 94.92% | 0.2040 |

---

### 🔥 关键发现：0.05 是一个**尖锐的峰值**

```
Accuracy vs Sync Weight (desync=0):

  97% ─┐              ⭐ 
       │             ╱╲ H2-c (0.05)
  96% ─┤            ╱  ╲
       │           ╱    ╲
  95% ─┤          ╱      ╲ H2-g (0.07)
       │         ╱        ╲
  94% ─┤  H2-f ─╱          ╲
       │  (0.03)            ╲
  93% ─┤                     ╲─ H2-e (0.1)
       └──┴──────┴──────┴──────┴──→ sync_weight
        0.03   0.05   0.07   0.1
```

| sync_weight | Acc | 与 H2-c 的差距 |
|:---:|:---:|:---:|
| 0.03 | 94.07% | -2.54% |
| **0.05** | **96.61%** | — |
| 0.07 | 94.92% | -1.69% |
| 0.10 | 93.22% | -3.39% |



# Base Model with: no sync + no adversarial

In [6]:
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --epochs 100 \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/base_chaotic \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/base_chaotic/base_chaotic.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

In [8]:
# 修复 chaotic_network.py 后
# model:
#   use_raw: false
#   use_differentiable_raw: false

# optional_modules:
#   differentiable_features:
#     enabled: false

# H2c but with seed = 43
# learning_rate: 0.0001
# dropout_rate: 0.5
# weight_decay: 0.01
# early_stop: patience: 999
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --epochs 100 \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

In [9]:
# H2c but with seed = 44
# with loss_sync
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --epochs 100 \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

In [10]:
# H2c but with seed = 43
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --epochs 300 \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path
^C
Traceback (most recent call last):
  File "/scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py", line 46, in <module>
    from chaotic_experiment import ChaoticExperiment, create_chaotic_experiments
  File "/scratch/project_2003370/yueyao/Model/experiments/chaotic_experiment.py", line 45, in <module>
    from base_experiment import BaseExperiment
  File "/scratch/project_2003370/yueyao/Model/experiments/base_experiment.py", line 32, in <module>
    from models.hybrid_models import TraditionalMLPBaseline, HybridModelManager
  File "/scratch/project_2003370/yueyao/Model/models/__init__.py", line 13, in <module>
    from .base_model import BaseModel, SklearnCompatibleModel, ModelConfig
  File "/scratch/project_2003370/yueyao/Model/models/base_model.py", line 36, in <module>
    from sklearn.base import BaseEstimator, ClassifierMixin
  File "/usr/local/lib64/python3.12/site-packages/sklearn/__init__.py

In [11]:
# H2c but with seed = 43
# learning_rate: 0.001
# --num_epochs 200
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path
^C
Traceback (most recent call last):
  File "/scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py", line 46, in <module>
    from chaotic_experiment import ChaoticExperiment, create_chaotic_experiments
  File "/scratch/project_2003370/yueyao/Model/experiments/chaotic_experiment.py", line 45, in <module>
    from base_experiment import BaseExperiment
  File "/scratch/project_2003370/yueyao/Model/experiments/base_experiment.py", line 32, in <module>
    from models.hybrid_models import TraditionalMLPBaseline, HybridModelManager
  File "/scratch/project_2003370/yueyao/Model/models/__init__.py", line 13, in <module>
    from .base_model import BaseModel, SklearnCompatibleModel, ModelConfig
  File "/scratch/project_2003370/yueyao/Model/models/base_model.py", line 36, in <module>
    from sklearn.base import BaseEstimator, ClassifierMixin
  File "/usr/local/lib64/python3.12/site-packages/sklearn/__init__.py

In [12]:
# H2c but with seed = 43
# learning_rate: 0.0001
# --num_epochs 200
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

^C
Traceback (most recent call last):
  File "/scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py", line 46, in <module>
    from chaotic_experiment import ChaoticExperiment, create_chaotic_experiments
  File "/scratch/project_2003370/yueyao/Model/experiments/chaotic_experiment.py", line 9, in <module>
    import matplotlib.pyplot as plt
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/pyplot.py", line 70, in <module>
    from matplotlib.figure import Figure, FigureBase, figaspect
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/figure.py", line 40, in <module>
    from matplotlib import _blocking_input, backend_bases, _docstring, projections
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/projections/__init__.py", line 55, in <module>
    from .. import axes, _docstring
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/axes/__init__.py", line 1, in <module>
    from . import _base
  File "/usr/local/lib64/python3.12/site-packa

In [13]:
# H2c but with seed = 43
# learning_rate: 0.0001
# dropout_rate: 0.5
# weight_decay: 0.01
# early_stop: patience: 999
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

^C
Traceback (most recent call last):
  File "/scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py", line 25, in <module>
    import torch
  File "/usr/local/lib64/python3.12/site-packages/torch/__init__.py", line 2611, in <module>
    from torch import _meta_registrations
  File "/usr/local/lib64/python3.12/site-packages/torch/_meta_registrations.py", line 12, in <module>
    from torch._decomp import (
  File "/usr/local/lib64/python3.12/site-packages/torch/_decomp/__init__.py", line 277, in <module>
    import torch._refs
  File "/usr/local/lib64/python3.12/site-packages/torch/_refs/__init__.py", line 1176, in <module>
    @_make_elementwise_binary_reference(
     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib64/python3.12/site-packages/torch/_refs/__init__.py", line 1062, in inner
    @elementwise_type_promotion_wrapper(
     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib64/python3.12/site-packages/torch/_prims_common/wrappers.py", line 126, in __c

In [14]:
# 修复 chaotic_network.py 后
# H2c but with seed = 43
# learning_rate: 0.0001
# dropout_rate: 0.5
# weight_decay: 0.01
# early_stop: patience: 999
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

^C
Traceback (most recent call last):
  File "/scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py", line 25, in <module>
    import torch
  File "/usr/local/lib64/python3.12/site-packages/torch/__init__.py", line 2611, in <module>
    from torch import _meta_registrations
  File "/usr/local/lib64/python3.12/site-packages/torch/_meta_registrations.py", line 12, in <module>
    from torch._decomp import (
  File "/usr/local/lib64/python3.12/site-packages/torch/_decomp/__init__.py", line 276, in <module>
    import torch._decomp.decompositions
  File "/usr/local/lib64/python3.12/site-packages/torch/_decomp/decompositions.py", line 16, in <module>
    import torch._prims as prims
  File "/usr/local/lib64/python3.12/site-packages/torch/_prims/__init__.py", line 1842, in <module>
    reshape = _make_prim(
              ^^^^^^^^^^^
  File "/usr/local/lib64/python3.12/site-packages/torch/_prims/__init__.py", line 321, in _make_prim
    prim_def = torch.library.custom_op(
           

In [15]:
# 修复 chaotic_network.py 后
# H2c but with seed = 43
# learning_rate: 0.0001
# dropout_rate: 0.5
# weight_decay: 0.01
# early_stop: patience: 999
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

In [16]:
# 修复 chaotic_network.py 后
# model:
#   use_raw: false
#   use_differentiable_raw: false

# optional_modules:
#   differentiable_features:
#     enabled: false

!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

In [17]:
# 修复 chaotic_network.py 后
# model:
#   use_raw: false
#   use_differentiable_raw: false

# optional_modules:
#   differentiable_features:
#     enabled: false

!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

^C
Traceback (most recent call last):
  File "/scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py", line 46, in <module>
    from chaotic_experiment import ChaoticExperiment, create_chaotic_experiments
  File "/scratch/project_2003370/yueyao/Model/experiments/chaotic_experiment.py", line 9, in <module>
    import matplotlib.pyplot as plt
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/pyplot.py", line 70, in <module>
    from matplotlib.figure import Figure, FigureBase, figaspect
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/figure.py", line 40, in <module>
    from matplotlib import _blocking_input, backend_bases, _docstring, projections
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/projections/__init__.py", line 55, in <module>
    from .. import axes, _docstring
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/axes/__init__.py", line 2, in <module>
    from ._axes import Axes
  File "/usr/local/lib64/python3.12/site-p

In [18]:
#  desync_weight
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

In [19]:
#  include_mfcc
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path
^C
Traceback (most recent call last):
  File "/scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py", line 46, in <module>
    from chaotic_experiment import ChaoticExperiment, create_chaotic_experiments
  File "/scratch/project_2003370/yueyao/Model/experiments/chaotic_experiment.py", line 45, in <module>
    from base_experiment import BaseExperiment
  File "/scratch/project_2003370/yueyao/Model/experiments/base_experiment.py", line 32, in <module>
    from models.hybrid_models import TraditionalMLPBaseline, HybridModelManager
  File "/scratch/project_2003370/yueyao/Model/models/__init__.py", line 13, in <module>
    from .base_model import BaseModel, SklearnCompatibleModel, ModelConfig
  File "/scratch/project_2003370/yueyao/Model/models/base_model.py", line 37, in <module>
    from sklearn.model_selection import train_test_split
  File "/usr/local/lib64/python3.12/site-packages/sklearn/model_selectio

In [20]:
# model:
#   include_mfcc: true            # 保持开启！(听觉特征)
#   # 注意：下面这两个参数通常由 optional_modules 控制，
#   # 确保你的 optional_modules 配置如下：

# optional_modules:
#   differentiable_features:
#     enabled: true               # <--- 必须改回 true！(开启 GPU 大脑)
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path
^C
Traceback (most recent call last):
  File "/scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py", line 46, in <module>
    from chaotic_experiment import ChaoticExperiment, create_chaotic_experiments
  File "/scratch/project_2003370/yueyao/Model/experiments/chaotic_experiment.py", line 45, in <module>
    from base_experiment import BaseExperiment
  File "/scratch/project_2003370/yueyao/Model/experiments/base_experiment.py", line 32, in <module>
    from models.hybrid_models import TraditionalMLPBaseline, HybridModelManager
  File "/scratch/project_2003370/yueyao/Model/models/__init__.py", line 13, in <module>
    from .base_model import BaseModel, SklearnCompatibleModel, ModelConfig
  File "/scratch/project_2003370/yueyao/Model/models/base_model.py", line 36, in <module>
    from sklearn.base import BaseEstimator, ClassifierMixin
  File "/usr/local/lib64/python3.12/site-packages/sklearn/__init__.py

In [21]:
# 这个版本已经集成了：
# 修复后的 MFCC 逻辑（正确初始化 + 正确融合）。
# 深层融合网络 (Fusion MLP)：解决“线性瓶颈”和“数值不匹配”问题。
# GPU/CPU 双模式支持：保留了可导重构和传统的 NumPy 提取。
# 完整的返回值处理：修复了 too many values to unpack 错误。

!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

^C
Traceback (most recent call last):
  File "/scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py", line 46, in <module>
    from chaotic_experiment import ChaoticExperiment, create_chaotic_experiments
  File "/scratch/project_2003370/yueyao/Model/experiments/chaotic_experiment.py", line 9, in <module>
    import matplotlib.pyplot as plt
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/pyplot.py", line 70, in <module>
    from matplotlib.figure import Figure, FigureBase, figaspect
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/figure.py", line 40, in <module>
    from matplotlib import _blocking_input, backend_bases, _docstring, projections
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/projections/__init__.py", line 55, in <module>
    from .. import axes, _docstring
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/axes/__init__.py", line 1, in <module>
    from . import _base
  File "/usr/local/lib64/python3.12/site-packa

In [22]:
# 这个版本已经集成了：
# 修复后的 MFCC 逻辑（正确初始化 + 正确融合）。
# 深层融合网络 (Fusion MLP)：解决“线性瓶颈”和“数值不匹配”问题。
# GPU/CPU 双模式支持：保留了可导重构和传统的 NumPy 提取。
# 完整的返回值处理：修复了 too many values to unpack 错误。

!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

^C
Traceback (most recent call last):
  File "/scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py", line 46, in <module>
    from chaotic_experiment import ChaoticExperiment, create_chaotic_experiments
  File "/scratch/project_2003370/yueyao/Model/experiments/chaotic_experiment.py", line 9, in <module>
    import matplotlib.pyplot as plt
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/pyplot.py", line 57, in <module>
    import matplotlib.colorbar
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/colorbar.py", line 19, in <module>
    from matplotlib import _api, cbook, collections, cm, colors, contour, ticker
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/contour.py", line 15, in <module>
    from matplotlib.backend_bases import MouseButton
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/backend_bases.py", line 49, in <module>
    from matplotlib import (
  File "/usr/local/lib64/python3.12/site-packages/matplotlib/text.py"

In [23]:
# 这个版本已经集成了：
# 修复后的 MFCC 逻辑（正确初始化 + 正确融合）。
# 深层融合网络 (Fusion MLP)：解决“线性瓶颈”和“数值不匹配”问题。
# GPU/CPU 双模式支持：保留了可导重构和传统的 NumPy 提取。
# 完整的返回值处理：修复了 too many values to unpack 错误。

!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0

In [24]:
!python /scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py \
    --system lorenz \
    --batch_size 32 \
    --output_dir  /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43 \
    --save_models \
    --model_types full_chaotic \
    --data_dir /scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2 \
    --config /scratch/project_2003370/yueyao/Model/experiments/configs/sync_experiments/H2/H2c_43/experiment_H2c_0.05.yaml

Fallback import setup: added /scratch/project_2003370/yueyao/Model to path

=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
^C
Traceback (most recent call last):
  File "/scratch/project_2003370/yueyao/Model/scripts/train_chaotic.py", line 46, in <module>
    from chaotic_experiment import ChaoticExperiment, create_chaotic_experiments
  File "/scratch/project_2003370/yueyao/Model/experiments/chaotic_experiment.py", line 45, in <module>
    from base_experiment import BaseExperiment
  File "/scratch/project_2003370/yue